# LLM Safety Evaluator & Governance
## Notebook 1 of 5 · Data Understanding & Anthropic Usage Policy Mapping

**Author:** Olivia Kim  · *Quantitative AI Risk Analyst* portfolio

This notebook characterises the harm surface of the Jigsaw corpus and emits the project's
centrepiece artefact: the **Jigsaw → Anthropic Usage Policy** mapping. Reusable logic lives
in `src/` (see [data_loader.py](../src/data_loader.py)); this notebook is the narrative.

> **Why characterise the data first?** You cannot set a safety threshold for a harm you
> have never measured. Understanding the distribution, the way harms co-occur, and the bias
> risks *is* the first control in the pipeline.

## 1 · Setup — load via `src` (reproducible, auditable)

In [ ]:
# --- repo bootstrap: make `from src import ...` work from notebooks/ ---
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.titleweight": "bold", "font.size": 10})

import src
from src import data_loader as dl
from src.data_loader import LABELS, ANTHROPIC_POLICY_MAP, SEVERITY_ORDER
print("src loaded. LABELS:", LABELS)

In [ ]:
df = dl.load_jigsaw()                 # integrity gates run inside load_jigsaw()
n = len(df)
print(f"Loaded {n:,} comments  ·  columns: {list(df.columns)}")
df[["id", "comment_text", *LABELS]].head(3)

## 2 · The 6-category distribution & class imbalance

> Safety data is extremely imbalanced. A model that predicts *everything clean* scores
> ~90% accuracy while catching **zero** harm — so accuracy is a trap, and the base rate of
> each harm drives our threshold and resampling choices downstream.

In [ ]:
prevalence = (df[LABELS].sum().to_frame("positive_count")
              .assign(prevalence_pct=lambda d: 100*d["positive_count"]/n)
              .sort_values("positive_count", ascending=False))
prevalence["severity"] = [ANTHROPIC_POLICY_MAP[i]["severity_tier"] for i in prevalence.index]
any_label = df[LABELS].sum(axis=1) > 0
n_flag = int(any_label.sum())
print(f"Flagged (>=1 label): {n_flag:,} ({100*n_flag/n:.1f}%)  |  "
      f"Clean: {n-n_flag:,} ({100*(n-n_flag)/n:.1f}%)")
print(f"==> predict-all-clean baseline = {100*(n-n_flag)/n:.1f}% accuracy, 0 harms caught.")
prevalence.style.format({"prevalence_pct": "{:.2f}%"})

In [ ]:
order = prevalence.index.tolist()
colors = [{"Critical":"#C44E52","High":"#DD8452","Medium":"#4C72B0"}[
            ANTHROPIC_POLICY_MAP[l]["severity_tier"]] for l in order]
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(order[::-1], prevalence.loc[order, "positive_count"][::-1], color=colors[::-1])
ax.set_title("Positive examples per category (bar colour = Anthropic severity tier)")
ax.set_xlabel("number of comments labelled positive")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for i, v in enumerate(prevalence.loc[order, "positive_count"][::-1]):
    ax.text(v, i, f" {v:,}", va="center", fontsize=8)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=v, label=k) for k, v in
                   {"Critical":"#C44E52","High":"#DD8452","Medium":"#4C72B0"}.items()],
          fontsize=8, title="severity")
plt.tight_layout(); plt.savefig(os.path.join(dl.results_dir(), "toxicity_distribution.png"),
                                bbox_inches="tight"); plt.show()

**Read-out.** The rarest categories — `threat` and `identity_hate` — are exactly the
**Critical**-severity ones. The harms we care most about have the thinnest data: a known
limitation carried into the threshold design (NB5).

## 3 · Multi-label co-occurrence — harm stacks

> Real abuse bundles harms; the classifier must treat categories as **correlated**, not
> independent. This justifies a hierarchical detect-`toxic`-then-specialise policy.

In [ ]:
flagged = df[any_label]
multi = (flagged[LABELS].sum(axis=1) > 1).mean()
print(f"Of flagged comments, {100*multi:.1f}% carry MORE THAN ONE label.")
co = pd.DataFrame(index=LABELS, columns=LABELS, dtype=float)
for a in LABELS:
    base = df[df[a] == 1]
    for b in LABELS:
        co.loc[a, b] = (base[b] == 1).mean() if len(base) else np.nan
fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(co.values.astype(float), cmap="Reds", vmin=0, vmax=1)
ax.set_xticks(range(6)); ax.set_xticklabels(LABELS, rotation=40, ha="right")
ax.set_yticks(range(6)); ax.set_yticklabels(LABELS)
ax.set_title("Conditional co-occurrence  P(column | row)"); ax.set_ylabel("given the comment is...")
for i in range(6):
    for j in range(6):
        v = co.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if v > 0.55 else "black", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); ax.grid(False)
plt.tight_layout(); plt.show()

**Read-out.** `toxic` is a *gateway/parent* label — nearly every other harm is also `toxic`; `identity_hate` and `insult` travel together.

## 4 · Comment length by category

> Length drives token cost (NB3 API spend) and is a potential **spurious shortcut**
> ("short ⇒ toxic") we must red-team against (NB4).

In [ ]:
clip = 2000
edges = np.linspace(0, clip, 61)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df.loc[~any_label, "char_len"].clip(upper=clip).to_numpy(), bins=edges,
             alpha=0.6, density=True, label="clean", color="#55A868")
axes[0].hist(df.loc[any_label, "char_len"].clip(upper=clip).to_numpy(), bins=edges,
             alpha=0.6, density=True, label="flagged", color="#C44E52")
axes[0].set_title("Comment length: clean vs flagged"); axes[0].set_xlabel(f"characters (clip {clip})")
axes[0].legend()
med = {l: df.loc[df[l]==1, "char_len"].median() for l in LABELS}; med["(clean)"] = df.loc[~any_label,"char_len"].median()
ml = pd.Series(med).sort_values()
axes[1].barh(ml.index, ml.values, color="#4C72B0"); axes[1].set_title("Median length by category")
axes[1].set_xlabel("median characters")
plt.tight_layout(); plt.show()
print("Flagged comments skew SHORTER -> 'short ⇒ toxic' spurious-shortcut risk (red-team in NB4).")

## 5 · Redacted sample comments per category

> We show **redacted** excerpts (`src.data_loader.redact`) — enough to see each harm's
> linguistic surface, nothing that republishes it. This mirrors a real T&S review surface
> and is itself a governance signal: a safety system should not reproduce the content it
> detects.

In [ ]:
rng = np.random.RandomState(42)
rows = []
for lab in LABELS:
    pool = df.index[df[lab] == 1]
    for idx in rng.choice(pool, size=min(2, len(pool)), replace=False):
        rows.append({"category": lab,
                     "other_labels": ", ".join(l for l in LABELS if l != lab and df.at[idx, l] == 1) or "-",
                     "chars": int(df.at[idx, "char_len"]),
                     "redacted_excerpt": dl.redact(df.at[idx, "comment_text"])})
pd.DataFrame(rows)

## 6 · ⭐ Mapping: Jigsaw 6 → Anthropic Usage Policy

> The project's centrepiece. Translates the dataset into the vocabulary a frontier-AI
> safety team governs in, with an EU AI Act touchpoint and a **severity tier** that drives
> thresholds (NB5), red-team priority (NB4), and drift tolerances (NB5). Full discussion in
> [docs/ANTHROPIC_USAGE_POLICY_MAPPING.md](../docs/ANTHROPIC_USAGE_POLICY_MAPPING.md).

In [ ]:
mapping = dl.policy_map_frame()
out_path = os.path.join(dl.outputs_dir(), "jigsaw_to_anthropic_usage_policy.csv")
mapping.to_csv(out_path, index=False)
print("Saved canonical policy mapping ->", out_path)
mapping.set_index("jigsaw_category")[["anthropic_usage_policy_clause","policy_harm_family",
                                      "eu_ai_act_touchpoint","severity_tier"]]

## 7 · Bias-risk scan — which identity groups appear, and where harm concentrates

> Jigsaw's follow-up competition existed because first-gen models learned to flag neutral
> sentences like *"I am a gay man"* as toxic. We **measure** identity-term prevalence and
> the conditional toxicity rate per group — an early bias signal, continuing the
> fair-lending thinking from Project 2. This is a *screening proxy to prioritise
> red-teaming*, not a bias verdict.

In [ ]:
bias = dl.identity_mentions(df)
base_tox = df["toxic"].mean() * 100
print(f"Corpus-wide toxic baseline: {base_tox:.2f}%")
xb = np.arange(len(bias))
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.bar(xb-0.2, bias["toxic_rate_given_mention_pct"], 0.4, label="toxic | mention", color="#C44E52")
ax.bar(xb+0.2, bias["identity_hate_rate_given_mention_pct"], 0.4, label="identity_hate | mention", color="#8172B3")
ax.axhline(base_tox, ls="--", color="gray", lw=1, label=f"corpus toxic baseline ({base_tox:.1f}%)")
ax.set_xticks(xb); ax.set_xticklabels(bias["identity_group"], rotation=20, ha="right")
ax.set_ylabel("% of comments mentioning the group"); ax.set_title("Conditional toxicity when an identity group is mentioned")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
bias.style.format({"pct_of_corpus":"{:.2f}%","toxic_rate_given_mention_pct":"{:.2f}%",
                   "identity_hate_rate_given_mention_pct":"{:.2f}%"})

**Governance action.** Identity mentions carry an above-baseline toxic rate — the pattern
that teaches a model the spurious *identity-term ⇒ toxic* association. This becomes a named
control objective: NB4 red-teams it (neutral identity sentences that must **not** flag), and
NB5 tracks a per-group false-positive-rate KRI.

## 8 · Summary

| Finding | Governance consequence | Carried into |
|---|---|---|
| ~90% clean (severe imbalance) | Accuracy misleading; recall-weighted, tiered metrics | NB2, NB5 |
| `toxic` is a gateway label; harms stack | Hierarchical, correlated detection | NB3, NB4 |
| Critical harms are rarest | Thinnest data on highest-stakes harms | NB5 |
| Flagged comments skew short | "short ⇒ toxic" shortcut risk | NB4 |
| Identity mentions raise conditional toxicity | Identity false-positive-bias risk | NB4, NB5 |
| **6 categories mapped to Anthropic Usage Policy** | Harm now speaks policy + regulation | **all downstream** |

*Next → **Notebook 2: Baseline ML** — the classical-ML control floor the Claude challenger must beat.*